# Phase 3-4: pretraining on 2x T4 (~15-20 h total over 2 sessions)
Accelerator: **GPU T4 x2**. Use **Save Version -> Save & Run All** so it runs in the background.
The script stops at 11.25 h, pushes a checkpoint to `$HF_USERNAME/minillm-125m-ckpt`, and
the next run of this notebook resumes automatically from it.

In [ ]:
# Kaggle: Add-ons -> Secrets: HF_TOKEN (write), HF_USERNAME, WANDB_API_KEY. Settings -> Internet ON.
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ["HF_TOKEN", "HF_USERNAME", "WANDB_API_KEY"]:
    try:
        os.environ[name] = secrets.get_secret(name)
    except Exception:
        print(f"secret {name} not set")

GITHUB_REPO = "https://github.com/<your-github-username>/minillm.git"  # <- edit
%cd /kaggle/working
!git clone -q {GITHUB_REPO} minillm 2>/dev/null || (cd minillm && git pull -q)
%cd /kaggle/working/minillm
!pip install -q -e .

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m MiniLLM.Training.Pretrain --config Configs/Run125M.yaml

## Quick checks once training is finished

In [ ]:
!python -m MiniLLM.Evaluation.Perplexity --model checkpoints/pretrain/latest --tokens-dir data/tokens
!python -m MiniLLM.Evaluation.HellaSwag --model checkpoints/pretrain/latest --tokenizer data/tokens/tokenizer